# Phase 05 · Bài 05 — Sentiment Analysis

**Mục tiêu:** xem sentiment như một bài phân loại văn bản cổ điển; tự viết Naive Bayes và logistic regression; đo bằng metric phù hợp; nhận diện các failure mode (phủ định, mỉa mai, lệch lớp).

## 1. Vì sao sentiment là bài học chuẩn

Sentiment nghe dễ: một câu nói tốt hay xấu thì gán nhãn. Nhưng gần như mọi câu dễ đều che một câu khó. `not great` đảo nghĩa bề mặt. `not bad at all` là khen dù chứa hai từ mang cực tính âm. `tight` trong review nhạc khác `tight` trong review thời trang. Mỉa mai (`oh great , another delay`) đảo nghĩa hoàn toàn mà không có dấu hiệu từ vựng ổn định. Emoji đôi khi mang nhiều tín hiệu hơn cả câu chữ.

Vì vậy sentiment là phòng thí nghiệm thu nhỏ của NLP cổ điển: mọi baseline đơn giản đều có một failure mode cụ thể, và mỗi mô hình phức tạp hơn ra đời để xử lý một failure mode trong số đó.

## 2. Công thức hai bước

1. **Biểu diễn**: biến văn bản thành vector đặc trưng (BoW, TF-IDF, n-gram, hoặc embedding).
2. **Phân loại**: fit một mô hình tuyến tính trên các ví dụ đã gán nhãn (Naive Bayes, logistic regression, SVM).

Điểm quan trọng: bước 1 quyết định mô hình *có thể* nhìn thấy tín hiệu gì. Nếu biểu diễn chỉ có unigram, mô hình không có cách nào phân biệt `good` và `not good` ngoài việc học trọng số riêng cho `not`; nếu biểu diễn có bigram `not_good`, mô hình có thể học một trọng số âm riêng cho cụm đó.

## 3. Multinomial Naive Bayes

Naive Bayes giả định mọi đặc trưng độc lập có điều kiện theo nhãn:

P(y | x) ∝ P(y) · Π P(x_i | y)

Với văn bản, P(x_i | y) được ước lượng bằng tần suất từ trong lớp cộng làm trơn Laplace:

P(w | y) = (count(w, y) + α) / (Σ_v count(v, y) + α·|V|)

Làm trơn cần thiết vì nếu một từ chưa từng xuất hiện trong lớp, xác suất bằng 0 và log biến thành −∞. `α = 1.0` là Laplace chuẩn; `α = 0.5` hoặc nhỏ hơn thường cho kết quả tốt hơn trên dữ liệu nhỏ.

Giả định độc lập là sai rõ ràng — `great` và `excellent` tương quan mạnh — nhưng kết quả vẫn tốt trên đặc trưng văn bản thưa. Lý do trực giác: với dữ liệu vừa phải, bộ phân loại chỉ cần biết mỗi từ nghiêng về phía nào nhiều hơn là ước lượng chính xác phân phối đồng thời.

Một hệ quả kỹ thuật ít được nói: NB cộng log xác suất của *mọi* token, kể cả token lặp, nên câu dài và câu ngắn không so sánh công bằng. Với corpus nhỏ, hiệu ứng độ dài có thể lấn át tín hiệu cảm xúc.

## 4. Logistic regression

Logistic regression bỏ giả định độc lập và học một trọng số thực cho từng đặc trưng:

P(y = 1 | x) = σ(w · x + b),  σ(z) = 1 / (1 + e^(−z))

Hàm mất mát là log loss cộng L2:

L = −(1/N) Σ [y log p + (1 − y) log(1 − p)] + λ‖w‖²

Trọng số âm là thứ Naive Bayes không biểu diễn được một cách tự nhiên. Với bigram `NOT_good`, logistic regression có thể học trọng số âm mạnh, trong khi Naive Bayes chỉ có thể dựa vào tần suất của chính bigram đó trong từng lớp.

## 5. Negation scoping

Khi không có bigram, một cách xử lý phủ định là gắn tiền tố `NOT_` cho các token sau từ phủ định cho đến dấu câu gần nhất:

```text
not good at all . but funny
→ not NOT_good NOT_at NOT_all . but funny
```

`good` và `NOT_good` trở thành hai đặc trưng khác nhau, nên mô hình có thể học trọng số ngược dấu. Đây là ba dòng tiền xử lý tạo ra chênh lệch đo được trên benchmark sentiment.

Cần lưu ý giới hạn: scoping theo dấu câu là quy ước thô. `not only good but excellent` sẽ bị xử lý sai; phủ định phạm vi xa (`I do not think this movie is good`) cũng khó. Scoping không tạo ra thông tin mới — nó chỉ tách một đặc trưng thành hai. Nếu dữ liệu huấn luyện không có ví dụ `NOT_x` ở cả hai lớp, mô hình không học được gì thêm.

## 6. Metric: đừng chỉ báo accuracy

Với dữ liệu lệch lớp, accuracy gây hiểu nhầm. Nếu 90% mẫu là positive, bộ phân loại luôn đoán positive đạt accuracy 0.9 nhưng vô dụng. Cần báo:

- **Precision / recall / F1 theo từng lớp.**
- **Macro-F1**: trung bình F1 của các lớp, trọng số bằng nhau. Đây là metric chính khi lớp lệch.
- **Weighted-F1**: trung bình theo tần suất lớp; hữu ích khi độ lệch lớp có ý nghĩa nghiệp vụ.
- **Confusion matrix**: số đếm thô, cho biết mô hình nhầm cặp lớp nào.
- **Mẫu lỗi**: đọc trực tiếp 5–10 dự đoán sai mỗi lớp. Không con số nào thay thế việc đọc lỗi.

Với lệch cực đoan (hơn 95–5), dùng AUROC và AUPRC; AUPRC nhạy hơn với lớp thiểu số.

## 7. Khi nào cần transformer

- Mỉa mai: mô hình cổ điển gần như chắc chắn thất bại.
- Review dài, cảm xúc đổi giữa chừng.
- Aspect-based sentiment: `camera was great but battery was terrible` cần gán cảm xúc cho từng khía cạnh.
- Ngôn ngữ ít tài nguyên: multilingual BERT cho baseline zero-shot.

Nếu không rơi vào các trường hợp trên, TF-IDF + bigram + negation scoping + logistic regression vẫn là baseline production hợp lý.

## 8. Bẫy tái lập

Số accuracy trong paper dùng split, preprocessing và tokenizer cụ thể. Nếu so mô hình mới với con số trong paper mà không chạy lại baseline trên đúng pipeline của mình, chênh lệch đo được sẽ vô nghĩa. Luôn tái tạo baseline trên pipeline hiện tại.


## A. Chạy code mẫu của bài

Cell dưới chạy `code/main.py` của bài 05 trong đúng thư mục repo, in exit code và toàn bộ stdout. Code mẫu huấn luyện Naive Bayes trên 5 review positive + 5 review negative, có negation scoping, rồi dự đoán 5 câu test và in metric.


In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "05-sentiment-analysis"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "Sentiment Analysis")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: Sentiment Analysis
Lệnh: phases\05-nlp-foundations-to-advanced\05-sentiment-analysis\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
[MISS] pred=+  true=-  :: this movie was not good
[OK] pred=+  true=+  :: loved every minute
[MISS] pred=+  true=-  :: terrible waste of time
[OK] pred=+  true=+  :: beautiful and brilliant
[OK] pred=-  true=-  :: i did not enjoy this at all

metrics: {'tp': 2, 'fp': 2, 'tn': 1, 'fn': 0, 'precision': 0.5, 'recall': 1.0, 'f1': 0.667}


## B. Thực hành riêng — bốn biến thể biểu diễn và một probe có kiểm soát

Ta tự viết lại pipeline (tokenize → negation scoping → bigram → Naive Bayes) để so bốn cấu hình đặc trưng trên cùng tập train/test:

1. unigram thuần,
2. unigram + negation scoping,
3. unigram + bigram,
4. unigram + bigram + negation scoping.

Bảng đầu cho biết chênh lệch thực tế trên 12 review huấn luyện. Với corpus nhỏ như vậy, thêm bigram không tự động tốt hơn: số tham số tăng, tần suất mỗi đặc trưng giảm, ước lượng nhiễu hơn.

Vì bảng đó trộn nhiều yếu tố, phần thứ hai là một **probe có kiểm soát**: chỉ dùng các cặp câu giống nhau trừ từ phủ định, cùng một mô hình, chỉ đổi cách biểu diễn. Probe này không phải benchmark; nó cô lập đúng cơ chế scoping. Nếu mô hình không scoping gán cùng một nhãn cho `the room was good` và `the room was not good`, ta thấy trực tiếp failure mode.


In [2]:
import math
import re
from collections import Counter

TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[.!?,;]")
NEGATION_WORDS = {"not", "no", "never", "nor", "none", "nothing", "neither", "n't"}
TERMINATORS = {".", "!", "?", ",", ";"}


def tokenize(text):
    return [t.lower() for t in TOKEN_RE.findall(text)]


def scope_negation(tokens):
    out, negate = [], False
    for token in tokens:
        if token in TERMINATORS:
            negate = False
            out.append(token)
        elif token in NEGATION_WORDS:
            negate = True
            out.append(token)
        else:
            out.append("NOT_" + token if negate else token)
    return out


def add_bigrams(tokens):
    return [a + "_" + b for a, b in zip(tokens, tokens[1:])]


def featurize(text, negation=False, bigram=False):
    tokens = tokenize(text)
    if negation:
        tokens = scope_negation(tokens)
    features = list(tokens)
    if bigram:
        features += add_bigrams(tokens)
    return features


TRAIN = [
    ("the food was great and the staff were friendly", "+"),
    ("absolutely loved every minute of it", "+"),
    ("beautiful cinematography and brilliant acting", "+"),
    ("not bad at all , surprisingly enjoyable", "+"),
    ("heartwarming , funny , and well written", "+"),
    ("great value and a warm welcome", "+"),
    ("the plot made no sense and the acting was terrible", "-"),
    ("not worth your time , would not recommend", "-"),
    ("boring and far too long", "-"),
    ("the battery dies fast and the screen is awful", "-"),
    ("i want my money back", "-"),
    ("service was slow and the room was dirty", "-"),
]

TEST = [
    ("the food was not good", "-"),
    ("i loved the story and the cast", "+"),
    ("terrible waste of two hours", "-"),
    ("surprisingly funny and warm", "+"),
    ("not worth the price", "-"),
    ("great value and friendly service", "+"),
    ("the movie was boring", "-"),
    ("a brilliant and moving film", "+"),
    ("the room was dirty and the service was slow", "-"),
    ("not bad at all", "+"),
]


def train_nb(docs, labels, alpha=0.5):
    vocab = sorted({feature for doc in docs for feature in doc})
    classes = sorted(set(labels))
    priors, likelihood = {}, {}
    for cls in classes:
        rows = [doc for doc, label in zip(docs, labels) if label == cls]
        priors[cls] = len(rows) / len(docs)
        counts = Counter(feature for doc in rows for feature in doc)
        denominator = sum(counts.values()) + alpha * len(vocab)
        likelihood[cls] = {f: (counts[f] + alpha) / denominator for f in vocab}
    return classes, priors, likelihood


def predict_nb(features, classes, priors, likelihood):
    scores = {}
    for cls in classes:
        score = math.log(priors[cls])
        for feature in features:
            if feature in likelihood[cls]:
                score += math.log(likelihood[cls][feature])
        scores[cls] = score
    return max(scores, key=scores.get)


def evaluate(y_true, y_pred, classes=("+", "-")):
    rows, macro = {}, 0.0
    for cls in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != cls and p == cls)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p != cls)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows[cls] = {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}
        macro += f1
    accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)
    return rows, accuracy, macro / len(classes)


def confusion(y_true, y_pred, classes=("+", "-")):
    table = {actual: {predicted: 0 for predicted in classes} for actual in classes}
    for t, p in zip(y_true, y_pred):
        table[t][p] += 1
    return table


y_true = [label for _, label in TEST]
EXPERIMENTS = [
    ("unigram", dict(negation=False, bigram=False)),
    ("unigram + negation", dict(negation=True, bigram=False)),
    ("unigram + bigram", dict(negation=False, bigram=True)),
    ("unigram + bigram + negation", dict(negation=True, bigram=True)),
]

print(f"{'biến thể':30s} {'accuracy':>9s} {'macro-F1':>9s}")
results = {}
for name, options in EXPERIMENTS:
    docs = [featurize(text, **options) for text, _ in TRAIN]
    labels = [label for _, label in TRAIN]
    classes, priors, likelihood = train_nb(docs, labels)
    y_pred = [predict_nb(featurize(text, **options), classes, priors, likelihood) for text, _ in TEST]
    rows, accuracy, macro = evaluate(y_true, y_pred)
    results[name] = (rows, accuracy, macro, y_pred)
    print(f"{name:30s} {accuracy:9.3f} {macro:9.3f}")

best_name = max(results, key=lambda key: results[key][2])
rows, accuracy, macro, y_pred = results[best_name]
print(f"\nChi tiết cho biến thể tốt nhất trên bộ test này: {best_name}")
print(f"{'lớp':>4s} {'precision':>10s} {'recall':>7s} {'F1':>6s} {'TP':>3s} {'FP':>3s} {'FN':>3s}")
for cls, row in rows.items():
    print(f"{cls:>4s} {row['precision']:10.3f} {row['recall']:7.3f} {row['f1']:6.3f} {row['tp']:3d} {row['fp']:3d} {row['fn']:3d}")
print(f"{'macro-F1':>22s} {macro:6.3f}   accuracy {accuracy:.3f}")

print("\nConfusion matrix (hàng = nhãn thật, cột = nhãn dự đoán):")
table = confusion(y_true, y_pred)
print(f"{'':>4s} {'+':>4s} {'-':>4s}")
for actual in ("+", "-"):
    print(f"{actual:>4s} {table[actual]['+']:4d} {table[actual]['-']:4d}")

print("\nTừng dự đoán:")
for (text, actual), predicted in zip(TEST, y_pred):
    mark = "OK  " if actual == predicted else "MISS"
    print(f"  [{mark}] true={actual} pred={predicted} :: {text}")

print("\n--- probe có kiểm soát: chỉ đổi từ phủ định ---")
PAIRS_TRAIN = [
    ("the food was good", "+"),
    ("the food was not good", "-"),
    ("the film was good", "+"),
    ("the film was not good", "-"),
    ("the food was bad", "-"),
    ("the food was not bad", "+"),
    ("the film was bad", "-"),
    ("the film was not bad", "+"),
]
PAIRS_TEST = [
    ("the room was good", "+"),
    ("the room was not good", "-"),
    ("the room was bad", "-"),
    ("the room was not bad", "+"),
]
for options in (dict(negation=False), dict(negation=True)):
    docs = [featurize(text, **options) for text, _ in PAIRS_TRAIN]
    labels = [label for _, label in PAIRS_TRAIN]
    classes, priors, likelihood = train_nb(docs, labels)
    pair_pred = [predict_nb(featurize(text, **options), classes, priors, likelihood) for text, _ in PAIRS_TEST]
    name = "unigram + negation" if options["negation"] else "unigram"
    pair_acc = sum(1 for (_, a), p in zip(PAIRS_TEST, pair_pred) if a == p) / len(PAIRS_TEST)
    print(f"{name:20s} accuracy={pair_acc:.3f}  dự đoán={pair_pred}")
    for (text, actual), predicted in zip(PAIRS_TEST, pair_pred):
        print(f"    true={actual} pred={predicted} :: {text}")
print("Mô hình không scoping gán cùng nhãn cho cặp good / not good và cặp bad / not bad.")


biến thể                        accuracy  macro-F1
unigram                            0.800     0.800
unigram + negation                 0.800     0.800
unigram + bigram                   0.700     0.697
unigram + bigram + negation        0.700     0.697

Chi tiết cho biến thể tốt nhất trên bộ test này: unigram
 lớp  precision  recall     F1  TP  FP  FN
   +      0.800   0.800  0.800   4   1   1
   -      0.800   0.800  0.800   4   1   1
              macro-F1  0.800   accuracy 0.800

Confusion matrix (hàng = nhãn thật, cột = nhãn dự đoán):
        +    -
   +    4    1
   -    1    4

Từng dự đoán:
  [OK  ] true=- pred=- :: the food was not good
  [MISS] true=+ pred=- :: i loved the story and the cast
  [MISS] true=- pred=+ :: terrible waste of two hours
  [OK  ] true=+ pred=+ :: surprisingly funny and warm
  [OK  ] true=- pred=- :: not worth the price
  [OK  ] true=+ pred=+ :: great value and friendly service
  [OK  ] true=- pred=- :: the movie was boring
  [OK  ] true=+ pred=+ :: a 

## C. Thí nghiệm — trọng số học được và dữ liệu lệch lớp

Hai quan sát bổ sung cho phần B.

**Trọng số.** Logistic regression trên cùng bộ đặc trưng (`unigram + bigram + negation`) cho biết từ/cụm nào đẩy dự đoán về phía positive hay negative. Ta in 5 trọng số âm nhất và 5 dương nhất. Đáng chú ý là hai đặc trưng cùng mang tiền tố `NOT_` có thể nhận trọng số ngược dấu, vì chúng xuất hiện trong hai lớp khác nhau. Đây là kiểm tra mô hình có học đúng tín hiệu hay chỉ học nhiễu của corpus nhỏ.

**Lệch lớp.** Một bộ test 9 positive + 1 negative cho thấy accuracy trông cao (0.9) trong khi mô hình luôn đoán positive. Macro-F1 phơi bày sự thật đó: F1 của lớp thiểu số bằng 0, nên trung bình chỉ còn khoảng 0.47. Đây chính là lỗi phổ biến khi chỉ báo accuracy.


In [3]:
import numpy as np

FEATURES = [featurize(text, negation=True, bigram=True) for text, _ in TRAIN]
VOCAB = sorted({feature for doc in FEATURES for feature in doc})
INDEX = {feature: i for i, feature in enumerate(VOCAB)}


def design_matrix(texts):
    X = np.zeros((len(texts), len(VOCAB)))
    for row, text in enumerate(texts):
        for feature in featurize(text, negation=True, bigram=True):
            column = INDEX.get(feature)
            if column is not None:
                X[row, column] += 1.0
    return X


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))


X = design_matrix([text for text, _ in TRAIN])
y = np.array([1.0 if label == "+" else 0.0 for _, label in TRAIN])
w = np.zeros(len(VOCAB))
b = 0.0
for _ in range(600):
    pred = sigmoid(X @ w + b)
    error = pred - y
    w -= 0.2 * (X.T @ error / len(y) + 0.01 * w)
    b -= 0.2 * error.mean()

ordered = sorted(zip(VOCAB, w), key=lambda pair: pair[1])
print("5 trọng số âm nhất (đẩy về negative):")
for feature, weight in ordered[:5]:
    print(f"  {feature:22s} {weight:+.3f}")
print("5 trọng số dương nhất (đẩy về positive):")
for feature, weight in ordered[-5:][::-1]:
    print(f"  {feature:22s} {weight:+.3f}")

print("\ncác đặc trưng NOT_ và trọng số:")
for feature, weight in ordered:
    if feature.startswith("NOT_"):
        print(f"  {feature:22s} {weight:+.3f}")

X_test = design_matrix([text for text, _ in TEST])
proba = sigmoid(X_test @ w + b)
y_pred_lr = ["+" if p >= 0.5 else "-" for p in proba]
rows_lr, accuracy_lr, macro_lr = evaluate(y_true, y_pred_lr)
print(f"\nlogistic regression: accuracy={accuracy_lr:.3f}  macro-F1={macro_lr:.3f}")
for (text, actual), p, predicted in zip(TEST, proba, y_pred_lr):
    mark = "OK  " if actual == predicted else "MISS"
    print(f"  [{mark}] p(+)={p:.3f} true={actual} pred={predicted} :: {text}")

print("\n--- dữ liệu lệch lớp ---")
imbalanced_true = ["+"] * 9 + ["-"]
majority_pred = ["+"] * 10
rows_imb, accuracy_imb, macro_imb = evaluate(imbalanced_true, majority_pred)
print(f"luôn đoán '+': accuracy={accuracy_imb:.3f}  macro-F1={macro_imb:.3f}")
print(f"  F1 lớp '+': {rows_imb['+']['f1']:.3f}   F1 lớp '-': {rows_imb['-']['f1']:.3f}")
print("Accuracy 0.9 không nói rằng mô hình phân biệt được hai lớp.")


5 trọng số âm nhất (đẩy về negative):
  and_far                -0.406
  boring                 -0.406
  boring_and             -0.406
  far                    -0.406
  far_too                -0.406
5 trọng số dương nhất (đẩy về positive):
  great                  +0.586
  ,                      +0.468
  cinematography_and     +0.351
  cinematography         +0.351
  brilliant_acting       +0.351

các đặc trưng NOT_ và trọng số:
  NOT_recommend          -0.277
  NOT_time               -0.277
  NOT_time_,             -0.277
  NOT_worth              -0.277
  NOT_worth_NOT_your     -0.277
  NOT_your               -0.277
  NOT_your_NOT_time      -0.277
  NOT_acting             -0.193
  NOT_acting_NOT_was     -0.193
  NOT_and                -0.193
  NOT_and_NOT_the        -0.193
  NOT_sense              -0.193
  NOT_sense_NOT_and      -0.193
  NOT_terrible           -0.193
  NOT_the                -0.193
  NOT_the_NOT_acting     -0.193
  NOT_was                -0.193
  NOT_was_NOT_terrible  

## D. Tự kiểm tra

1. Vì sao Naive Bayes vẫn hoạt động dù giả định độc lập sai?
2. Làm trơn Laplace bảo vệ mô hình khỏi lỗi cụ thể nào?
3. Negation scoping khác bigram `not_good` ở điểm nào, và khi nào nên dùng cách nào?
4. Vì sao probe ở phần B cho cùng một dự đoán cho `good` và `not good` khi không scoping?
5. Vì sao macro-F1 phù hợp hơn accuracy khi lớp lệch?
6. Ba trường hợp nào buộc phải chuyển sang transformer?

**Giới hạn:** corpus ở đây rất nhỏ và do người viết tự tạo. Bảng biến thể ở phần B là quan sát trên 12 review, không phải benchmark; probe chỉ cô lập cơ chế phủ định. Con số accuracy/macro-F1 không phải ước lượng chất lượng sentiment ngoài đời. Mọi so sánh với baseline trong paper phải chạy lại trên cùng pipeline.
